# SigLIP 2 Vision-Language Pipeline — DIMER tutorial

**Profile:** `MULTI-CAPABILITY` · **Notebook spec:** v1.0

This notebook uses the pinned `google/siglip2-base-patch16-224` checkpoint through
the repository's public API for zero-shot classification, image/text embeddings,
cosine similarity, and text-to-image retrieval. **No gradient training, fine-tuning,
in-context conditioning, or fitted preprocessing state occurs.** Upstream provides
the pretrained model/processor; this repository adds immutable pinning, integrity
verification, safe local loading, stable inference contracts, exports, and provenance.

Outputs are uncalibrated inference/ranking evidence. This notebook does **not**
provide object detection, semantic segmentation, OCR, caption generation, calibrated
probabilities, universal thresholds, or production serving.

## Prerequisites

Python 3.12 is the release reference. The frozen lock installs the official
**CPU-only** PyTorch wheel, so this tutorial intentionally runs on CPU even on a GPU
host. GPU execution requires a separately pinned/tested environment and is outside
this release contract. First use may need network access to fetch the immutable
checkpoint; the verified weight file is 1,500,800,904 bytes.

Default data are deterministic **synthetic tutorial/smoke assets**, not benchmark
data. BYOD is optional and disabled by default. The pipeline converts images to RGB,
uses the pinned 224×224 image contract, lowercases model-bound text, and uses a
64-token text maximum. Candidate labels/queries must be non-empty. Uploaded images
remain in the notebook runtime; the pipeline rejects HTTP(S) image URLs and does not
send image contents to a hosted inference API.


## 1. Bootstrap the frozen CPU environment


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/siglip2-vision-language-pipeline.git"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path("/content/siglip2-vision-language-pipeline")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.lock.txt"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", "."],
    check=True,
)
print("Repository root:", ROOT.resolve())


## 2. Runtime and immutable model identity

The lock is the validated Linux/CPU reference graph. No mixed precision,
quantization, compilation, stochastic decoding, random split, or random initialization
is used. Small floating-point differences can still occur across builds/platforms.


In [ ]:
import platform
from importlib.metadata import version

import numpy as np
import torch
from PIL import Image

from siglip2_pipeline import (
    MODEL_ID,
    MODEL_REVISION,
    build_provenance,
    load_pipeline,
    write_provenance,
)

DEVICE = "cpu"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", version("transformers"))
print("Device:", DEVICE)

pipe = load_pipeline(device=DEVICE)
prov = build_provenance(pipeline=pipe)
print("Model ID:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Checkpoint source:", prov["model"].get("checkpoint_source"))
print("Manifest verified:", prov["model"].get("manifest_verified"))
print("Weight SHA-256:", prov["model"]["weight_sha256"])
print("Weight bytes:", prov["model"]["weight_size_bytes"])


## 3–8. Default sample, classification, embeddings, similarity, retrieval, evaluation

Three generated RGB images provide deterministic tutorial ground truth. Zero-shot
scores are independent SigLIP sigmoids: **not calibrated probabilities** and not
required to sum to one. The tutorial uses argmax only for its sanity check. Any
deployment threshold/abstention rule belongs to the **downstream application** and
must be calibrated on representative labelled data.

Embeddings are L2-normalized representations, not predictions, and have no intrinsic
accuracy metric. Similarity/retrieval use cosine similarity; their usefulness requires
a downstream labelled evaluation. The reported top-1 accuracy and recall@1 are tiny
synthetic sanity metrics, compared with a fixed-class 1/3 classification baseline;
they are not estimates of generalization.


In [ ]:
import hashlib
from dataclasses import asdict

SAMPLE_DIR = ROOT / "outputs" / "sample-data"
subprocess.run(
    [sys.executable, "examples/sample-data/generate_samples.py", "--output-dir", str(SAMPLE_DIR)],
    check=True,
)
images = [SAMPLE_DIR / n for n in ("red_square.ppm", "green_circle.ppm", "blue_triangle.ppm")]
expected_labels = ["red square", "green circle", "blue triangle"]
candidate_labels = [*expected_labels, "abstract geometric shape"]

for path in images:
    with Image.open(path) as im:
        print(path.name, im.mode, im.size, hashlib.sha256(path.read_bytes()).hexdigest())

classification_rows, correct = [], 0
for image_path, expected in zip(images, expected_labels, strict=True):
    scores = pipe.zero_shot_classify(image_path, candidate_labels)
    predicted = scores[0].label
    correct += int(predicted == expected)
    classification_rows.append({
        "image": image_path.name, "expected_label": expected,
        "predicted_label": predicted, "scores": [asdict(x) for x in scores],
    })
top1_sanity_accuracy = correct / len(images)
fixed_class_baseline_accuracy = 1 / len(expected_labels)
print("Top-1 sanity accuracy:", top1_sanity_accuracy)
print("Fixed-class baseline:", fixed_class_baseline_accuracy)

image_embeddings = pipe.embed_image(images)
text_embeddings = pipe.embed_text(expected_labels)
similarity = pipe.similarity(images, expected_labels)
print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape)
print("Similarity rows:", [p.name for p in images])
print("Similarity columns:", expected_labels)
print(np.array2string(similarity, precision=4))

retrieval_rows, hits1 = [], 0
for query, expected_image in zip(expected_labels, images, strict=True):
    hits = pipe.retrieve(query, images, top_k=len(images))
    hits1 += int(images[hits[0].index].name == expected_image.name)
    retrieval_rows.append({
        "query": query, "expected_image": expected_image.name,
        "hits": [{"rank": r, "index": h.index, "filename": images[h.index].name, "score": h.score}
                 for r, h in enumerate(hits, 1)],
    })
retrieval_recall_at_1 = hits1 / len(expected_labels)
print("Retrieval recall@1:", retrieval_recall_at_1)


## 9. Optional BYOD / new user image

Set `ENABLE_BYOD=True` to test one genuinely new user image. Colab uses its upload
dialog; other Jupyter environments use `BYOD_PATH`. The file must exist and be
Pillow-decodable before inference. Edit `BYOD_LABELS` for the intended domain.


In [ ]:
ENABLE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}
BYOD_LABELS = ["flooded street", "normal road", "fallen electrical pole"]
byod_image = None

if ENABLE_BYOD:
    try:
        from google.colab import files as colab_files  # type: ignore
        uploaded = colab_files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one image.")
        name, data = next(iter(uploaded.items()))
        target = ROOT / "outputs" / "byod" / Path(name).name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    except ModuleNotFoundError:
        if not BYOD_PATH:
            raise RuntimeError("Outside Colab, set BYOD_PATH before enabling BYOD.") from None
        target = Path(BYOD_PATH).expanduser()

    if not target.is_file():
        raise FileNotFoundError(target)
    try:
        with Image.open(target) as im:
            im.verify()
    except Exception as exc:
        raise ValueError(f"BYOD file is not a decodable image: {target}") from exc
    if not BYOD_LABELS or any(not x.strip() for x in BYOD_LABELS):
        raise ValueError("BYOD_LABELS must contain non-empty labels.")

    byod_image = target
    byod_scores = pipe.zero_shot_classify(target, BYOD_LABELS)
    byod_embedding = pipe.embed_image([target])
    print("BYOD top label:", byod_scores[0].label)
else:
    print("BYOD disabled; default path is non-interactive.")


## 10. Default new-data inference

A deterministic `yellow_square.ppm`, distinct from the three-image evaluation set,
exercises the new-input path without interaction. It remains synthetic evidence and
does not establish deployment accuracy.


In [ ]:
NEW_DATA_DIR = ROOT / "outputs" / "new-data"
NEW_DATA_DIR.mkdir(parents=True, exist_ok=True)
new_image_path = NEW_DATA_DIR / "yellow_square.ppm"
im = Image.new("RGB", (32, 32), "white")
px = im.load()
for y in range(8, 24):
    for x in range(8, 24):
        px[x, y] = (255, 255, 0)
im.save(new_image_path)

new_labels = ["yellow square", "blue circle", "red triangle", "abstract geometric shape"]
new_data_scores = pipe.zero_shot_classify(new_image_path, new_labels)
print("New-data top label:", new_data_scores[0].label)


## 11. Machine-readable outputs

Every demonstrated capability gets a stable export. Embedding archives store input
identifiers beside vectors. Provenance records the effective model, immutable
revision, verified checkpoint, runtime packages, CPU device, and inference semantics.
No trained/adapted model artifact is produced because this is pretrained inference.


In [ ]:
import csv
import json

OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
(OUTPUT / "classification.json").write_text(json.dumps(classification_rows, indent=2) + "\n")
(OUTPUT / "retrieval.json").write_text(json.dumps(retrieval_rows, indent=2) + "\n")
(OUTPUT / "new_data_classification.json").write_text(
    json.dumps(
        {"image": new_image_path.name, "scores": [asdict(x) for x in new_data_scores]},
        indent=2,
    )
    + "\n"
)
with (OUTPUT / "similarity.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["image", *expected_labels])
    for p, row in zip(images, similarity, strict=True):
        w.writerow([p.name, *map(float, row)])

np.savez_compressed(
    OUTPUT / "image_embeddings.npz",
    image_ids=np.asarray([p.name for p in images]), vectors=image_embeddings,
)
np.savez_compressed(
    OUTPUT / "text_embeddings.npz",
    text_ids=np.asarray(expected_labels), vectors=text_embeddings,
)
metrics = {
    "evidence_type": "synthetic_tutorial_sanity_only",
    "sample_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in images},
    "classification": {"metric": "top1_accuracy", "value": top1_sanity_accuracy,
                       "fixed_class_baseline": fixed_class_baseline_accuracy},
    "retrieval": {"metric": "recall_at_1", "value": retrieval_recall_at_1},
}
(OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")

if ENABLE_BYOD and byod_image is not None:
    (OUTPUT / "byod_classification.json").write_text(
        json.dumps(
            {"image": byod_image.name, "scores": [asdict(x) for x in byod_scores]},
            indent=2,
        )
        + "\n"
    )
    np.savez_compressed(
        OUTPUT / "byod_image_embedding.npz",
        image_ids=np.asarray([byod_image.name]), vectors=byod_embedding,
    )

write_provenance(OUTPUT / "provenance.json", pipeline=pipe)
expected_outputs = [
    "classification.json", "image_embeddings.npz", "text_embeddings.npz",
    "similarity.csv", "retrieval.json", "metrics.json",
    "new_data_classification.json", "provenance.json",
]
missing = [n for n in expected_outputs if not (OUTPUT / n).is_file()]
if missing:
    raise RuntimeError(f"Missing expected tutorial outputs: {missing}")
print("Wrote:", expected_outputs)


## Troubleshooting

Integrity/download failures must be fixed rather than bypassing pin/hash checks.
For memory pressure, reduce images per call. A GPU host is expected to remain on CPU
because the release lock is CPU-only. For BYOD failures, verify the path/upload and
image decodability. Unexpected scores should first trigger review of prompt/label
wording; SigLIP scores are prompt-dependent and uncalibrated.

## Interpretation, limits, and next steps

A successful run proves that the frozen CPU reference environment installs, the
pinned checkpoint passes integrity checks, all five public operations execute, and
machine-readable outputs/provenance are produced. It **does not** prove production
fitness, domain accuracy, fairness, robustness, calibration, latency, GPU
compatibility, or universal thresholds.

For deployment, evaluate representative labelled real images, define the production
prompt/label policy, inspect failure modes/subgroups, calibrate thresholds or
abstention rules, and benchmark the intended serving hardware. Useful next steps are
BYOD with non-sensitive data, prompt comparisons, labelled real-image retrieval, and
downstream evaluation of exported embeddings.
